# RAGAS evaluation with cost analysis

This notebook evaluates open-source Fireworks AI powered RAG app against an OpenAI gpt-4.1-mini powered equivalent. It compares cost via LangSmith

# Environment Setup

In [1]:
from eval.generate_testset import generate_testset
from eval.runner import run_rag_over_testset
from app.rag import get_rag_graph
import os, pandas as pd
from dotenv import load_dotenv
from langsmith import tracing_context
from eval.ragas_scoring import score_rag_rows_sync, summarize_scores, compare_providers

load_dotenv()

os.environ["LANGSMITH_PROJECT"] = os.environ["LANGSMITH_PROJECT_FIREWORKS"]
os.environ["LANGSMITH_TRACING"] = "true"

# Generate or Load Test Set

Generate Test set with RAGAS

In [2]:
testset_df = generate_testset(data_dir="data", test_size=3)

# Display test set
testset_df


Applying SummaryExtractor:   0%|          | 0/40 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/40 [00:00<?, ?it/s]

/Users/saurabh3981/Desktop/agenticai/code/AIEC1/10_LLM_Servers/.venv/lib/python3.13/site-packages/ragas/testset/transforms/base.py:198: UserWarning: Using sync embedding model OpenAIEmbeddings in async context. This may impact performance. Consider using an async-compatible embedding model for better performance.
  property_name, property_value = await self.extract(node)


Applying ThemesExtractor:   0%|          | 0/40 [00:00<?, ?it/s]

Applying NERExtractor:   0%|          | 0/40 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/3 [00:00<?, ?it/s]

,user_input,reference,reference_contexts
0,Who is Paula Plummer in the 2021 AAHA/AAFP Fel...,"Paula Plummer, LVT, VTS (ECC, SAIM), is one of...",[VETERINARY PRACTICE GUIDELINES\n2021 AAHA/AAF...
1,how feline retrovirus testing and management l...,The 2020 AAFP feline retrovirus testing and ma...,[<1-hop>\n\nguidelines. J Feline Med Surg 2019...
2,What are the recommended litter box management...,"For kittens, it is recommended to discuss appr...",[<1-hop>\n\nassume these behaviors are normal ...


# Run Fireworks RAG Pipeline (with LangSmith)

In [3]:
with tracing_context(project_name=os.environ["LANGSMITH_PROJECT_FIREWORKS"]):
    firework_graph = get_rag_graph("fireworks")
    firework_eval_result = run_rag_over_testset(firework_graph, testset_df)


firework_eval_result[0]

{'user_input': 'Who is Paula Plummer in the 2021 AAHA/AAFP Feline Life Stage Guidelines?',
 'reference': 'Paula Plummer, LVT, VTS (ECC, SAIM), is one of the authors of the 2021 AAHA/AAFP Feline Life Stage Guidelines and is affiliated with the Texas A&M University Veterinary Medical Teaching Hospital, College Station, Texas, USA.',
 'reference_contexts': ['VETERINARY PRACTICE GUIDELINES\n2021 AAHA/AAFP Feline Life Stage Guidelines*\nJessica Quimby, DVM, PhD, DACVIMy, Shannon Gowland, DVM, DABVPy, Hazel C. Carney, DVM, MS, DABVP,\nTheresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,\nDVM, PhD, DACVIM\nABSTRACT\nThe guidelines, authored by a Task Force of experts in feline clinical medicine, are an update and extension of the AAFP–AAHA\nFeline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in the Journal of Feline Medicine and\nSurgery (volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and t

# Run OpenAI RAG Pipeline (with LangSmith)

In [4]:
with tracing_context(project_name=os.environ["LANGSMITH_PROJECT_OPENAI"]):
    openai_graph = get_rag_graph("openai")
    openai_eval_result = run_rag_over_testset(openai_graph, testset_df)


openai_eval_result[0]


{'user_input': 'Who is Paula Plummer in the 2021 AAHA/AAFP Feline Life Stage Guidelines?',
 'reference': 'Paula Plummer, LVT, VTS (ECC, SAIM), is one of the authors of the 2021 AAHA/AAFP Feline Life Stage Guidelines and is affiliated with the Texas A&M University Veterinary Medical Teaching Hospital, College Station, Texas, USA.',
 'reference_contexts': ['VETERINARY PRACTICE GUIDELINES\n2021 AAHA/AAFP Feline Life Stage Guidelines*\nJessica Quimby, DVM, PhD, DACVIMy, Shannon Gowland, DVM, DABVPy, Hazel C. Carney, DVM, MS, DABVP,\nTheresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,\nDVM, PhD, DACVIM\nABSTRACT\nThe guidelines, authored by a Task Force of experts in feline clinical medicine, are an update and extension of the AAFP–AAHA\nFeline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in the Journal of Feline Medicine and\nSurgery (volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and t

## Side-by-side response preview

Compare Fireworks and OpenAI answers on the same test questions before RAGAS scoring.

In [5]:
comparison_df = pd.DataFrame(
    {
        "user_input": [row["user_input"] for row in firework_eval_result],
        "reference": [row["reference"] for row in firework_eval_result],
        "fireworks_response": [row["response"] for row in firework_eval_result],
        "openai_response": [row["response"] for row in openai_eval_result],
        "fireworks_retrieved_chunks": [
            len(row["retrieved_contexts"]) for row in firework_eval_result
        ],
        "openai_retrieved_chunks": [
            len(row["retrieved_contexts"]) for row in openai_eval_result
        ],
    }
)

comparison_df

,user_input,reference,fireworks_response,openai_response,fireworks_retrieved_chunks,openai_retrieved_chunks
0,Who is Paula Plummer in the 2021 AAHA/AAFP Fel...,"Paula Plummer, LVT, VTS (ECC, SAIM), is one of...",Paula Plummer is one of the authors of the 202...,"Paula Plummer, LVT, VTS (ECC, SAIM), is one of...",4,4
1,how feline retrovirus testing and management l...,The 2020 AAFP feline retrovirus testing and ma...,The excerpt lists **the 2020 AAFP Feline Retro...,Feline retrovirus testing and management and f...,4,4
2,What are the recommended litter box management...,"For kittens, it is recommended to discuss appr...",**Litter‑box management for kittens (to keep t...,The recommended litter box management practice...,4,4


## RAGAS Scoring

In [6]:
# Score Fireworks rows
fireworks_scores = score_rag_rows_sync(firework_eval_result)

# Summarize scores
summarize_scores(fireworks_scores)


context_recall     0.833333
faithfulness       0.777778
answer_accuracy    0.750000
dtype: float64

In [7]:
# Score OpenAI rows
openai_scores = score_rag_rows_sync(openai_eval_result)

# Summarize scores
summarize_scores(openai_scores)


context_recall     0.833333
faithfulness       0.666667
answer_accuracy    0.666667
dtype: float64

## compare scores

In [8]:
compare_providers(fireworks_scores, openai_scores)

,fireworks,openai
context_recall,0.833333,0.833333
faithfulness,0.777778,0.666667
answer_accuracy,0.750000,0.666667


## LangSmith Cost Analysis

Fill in `estimated_cost_usd` and `notes` from the [LangSmith dashboard](https://smith.langchain.com) for each project. Token counts are pulled automatically when traces exist.

In [9]:
from langsmith import Client


def _project_token_stats(project_name: str) -> dict[str, int | str | None]:
    try:
        client = Client()
        runs = list(client.list_runs(project_name=project_name, limit=100))
        return {
            "run_count": len(runs),
            "prompt_tokens": sum(r.prompt_tokens or 0 for r in runs),
            "completion_tokens": sum(r.completion_tokens or 0 for r in runs),
            "total_tokens": sum(r.total_tokens or 0 for r in runs),
            "notes": "",
        }
    except Exception as exc:
        return {
            "run_count": None,
            "prompt_tokens": None,
            "completion_tokens": None,
            "total_tokens": None,
            "notes": f"Could not load runs: {exc}",
        }


cost_rows = []
for provider, env_key in [
    ("Fireworks", "LANGSMITH_PROJECT_FIREWORKS"),
    ("OpenAI", "LANGSMITH_PROJECT_OPENAI"),
]:
    project_name = os.environ[env_key]
    stats = _project_token_stats(project_name)
    cost_rows.append(
        {
            "provider": provider,
            "project_name": project_name,
            "run_count": stats["run_count"],
            "prompt_tokens": stats["prompt_tokens"],
            "completion_tokens": stats["completion_tokens"],
            "total_tokens": stats["total_tokens"],
            "estimated_cost_usd": None,
            "notes": stats["notes"],
        }
    )

cost_analysis_df = pd.DataFrame(cost_rows)
cost_analysis_df

,provider,project_name,run_count,prompt_tokens,completion_tokens,total_tokens,estimated_cost_usd,notes
0,Fireworks,llm-servers-fireworks-rag,100,46176,7620,53796,None,
1,OpenAI,llm-servers-openai-rag,24,43352,2288,45640,None,
